# 22 - Train a DQN Model Offline with Position Embeddings

This notebook follows [02 - Offline DQN](02_train_offline_dqn.ipynb): prepare replay,
tokenize events, build an encoder and Q head, train, and save locally. It adds elapsed
time using Mouse Core's optional **numeric position**.

An input field opts in with three settings:

- `position_field` names the payload value to add to its tokens.
- `position_fourier_min` sets the lower bound for scaling.
- `position_fourier_max` sets the upper bound for scaling.

The bounds map values to `[-1, 1]` without clipping. Missing or `None` values add
nothing; zero and negative values are supported. Every token generated by the input
field receives the same position. No extra token is emitted. Every input field must name
its provider explicitly.

This example uses elapsed seconds, but a position can be any finite numeric context
value. For real timestamps, subtract the actual episode start time. The toy replay below
is simulated; this notebook demonstrates the API rather than establishing performance on
real data.


In [ ]:
import random
from copy import deepcopy
from pathlib import Path

import torch
from torch import nn

from mouse_core import AdamW, polyak_update
from mouse_core.data import (
    DataLoader,
    EventLog,
    Tokenizer,
    save_tokenizer,
)
from mouse_core.models import (
    MLP,
    Backbone,
    DecodeCache,
    Embedding,
    Model,
    MultiEmbedding,
    prepare_decode,
    save_model,
)
from mouse_core.objectives import DqnObjective, boundary_discount

# Model
PRETRAINED = "Qwen/Qwen3-0.6B"
MAX_ACTIONS = 2
POSITION_NUM_FREQUENCIES = 16
HEAD_LAYERS = 1
HEAD_SCALE = 0.1

# Position scaling, in elapsed seconds
POSITION_FOURIER_MIN = 0.0
POSITION_FOURIER_MAX = 60.0

# Training
TOKEN_BUDGET = 2048
NUM_CYCLES = 2
TRAIN_STEPS = 10

# Optimizer learning rates
LR_EMBEDDING = 1e-5
LR_BACKBONE = 1e-5
LR_HEADS = 1e-5

# Target interpolation rates
POLYAK_TAU_EMBEDDING = 0.1
POLYAK_TAU_BACKBONE = 0.1
POLYAK_TAU_VALUE = 0.001

# Output and runtime
OUTPUT_DIR = Path("outputs/offline_dqn_time")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

## Load Data

The observation stays the same throughout this two-action toy task. Action 0 receives a
reward before 10 elapsed seconds; action 1 receives a reward from 10 seconds onward.
Random delays make elapsed time differ from decision index.

Each episode starts at zero seconds. Agent events carry the time of the command, while
environment events carry the later time of the result.


In [ ]:
def make_replay(*, episodes=32, decisions=6, seed=0):
    rng = random.Random(seed)
    log = EventLog(name="timed-toy-replay")

    for episode in range(episodes):
        seconds = 0
        log.event(
            provider="coordinator",
            episode_index=episode,
            step_index=0,
            payload={"episode_start": True},
        )
        log.event(
            provider="environment",
            episode_index=episode,
            step_index=0,
            payload={
                "observation": "same sensor reading",
                "elapsed_seconds": seconds,
                "action_requested": True,
            },
        )

        for step in range(1, decisions + 1):
            action = rng.randrange(MAX_ACTIONS)
            reward = float(action == int(seconds >= 10))
            log.event(
                provider="agent",
                episode_index=episode,
                step_index=step,
                payload={
                    "action": action,
                    "elapsed_seconds": seconds,
                },
            )

            seconds += rng.choice((1, 3, 7))
            payload = {
                "observation": "same sensor reading",
                "elapsed_seconds": seconds,
                "reward": reward,
            }
            if step == decisions:
                payload["terminated"] = True
            else:
                payload["action_requested"] = True

            log.event(
                provider="environment",
                episode_index=episode,
                step_index=step,
                payload=payload,
            )

    return log


replay = make_replay()
stores = [replay.to_store()]

## Data Pipeline

Each input field explicitly selects its provider, input value, and position. The example
applies elapsed time to all four input fields. Omitting the three position settings
leaves that field untimed.

The tokenizer stores normalized position values alongside token IDs. The separate
`step_index` still groups decisions and aligns DQN targets.


In [ ]:
tokenizer = Tokenizer(
    tokenizers={"text": PRETRAINED},
    step_field="step_index",
    episode_field="episode_index",
    input_fields=[
        {
            "type": "text",
            "tokenizer": "text",
            "provider": "agent",
            "input_field": "action",
            "format": 'action={action},',
            "position_field": "elapsed_seconds",
            "position_fourier_min": POSITION_FOURIER_MIN,
            "position_fourier_max": POSITION_FOURIER_MAX,
        },
        {
            "type": "text",
            "tokenizer": "text",
            "provider": "environment",
            "input_field": "observation",
            "format": 'obs={observation},',
            "position_field": "elapsed_seconds",
            "position_fourier_min": POSITION_FOURIER_MIN,
            "position_fourier_max": POSITION_FOURIER_MAX,
        },
        {
            "type": "text",
            "tokenizer": "text",
            "provider": "environment",
            "input_field": "reward",
            "format": 'reward={reward:g},',
            "position_field": "elapsed_seconds",
            "position_fourier_min": POSITION_FOURIER_MIN,
            "position_fourier_max": POSITION_FOURIER_MAX,
        },
        {
            "type": "token",
            "tokenizer": "text",
            "provider": "environment",
            "input_field": "action_requested",
            "format": "\n",
            "head_output": True,
            "position_field": "elapsed_seconds",
            "position_fourier_min": POSITION_FOURIER_MIN,
            "position_fourier_max": POSITION_FOURIER_MAX,
        },
    ],
    objective_fields=[
        {"input_field": "action", "missing_value": None},
        {"input_field": "reward", "missing_value": 0.0},
        {"input_field": "terminated", "missing_value": False},
        {"input_field": "truncated", "missing_value": False},
    ],
)

def sample_ranges(dataset):
    """Keep all events belonging to one outer episode together."""
    start, previous = 0, None
    for index, episode in enumerate(dataset["episode_index"][:]):
        if episode is not None and episode != previous:
            if index:
                yield start, index
            start, previous = index, episode
    if len(dataset):
        yield start, len(dataset)


def sample_eligible(events):
    """Train only on episodes with a recorded final environment result."""
    last = events[-1]
    return last["provider"] == "environment" and (
        last["payload"].get("terminated") is True
        or last["payload"].get("truncated") is True
    )


loader = DataLoader(
    stores=stores,
    token_budget=TOKEN_BUDGET,
    sample_ranges=sample_ranges,
    sample_eligible=sample_eligible,
    transform=tokenizer.encode,
    prefetch=4,
    num_workers=4,
)

# Every token from this field receives the same normalized position value.
probe = {
    "provider": "environment",
    "payload": {
        "observation": "same sensor reading",
        "elapsed_seconds": 7,
    },
}
timed = tokenizer(probe)
untimed = tokenizer(
    {
        "provider": "environment",
        "payload": {"observation": "same sensor reading"},
    }
)
normalized_time = (
    2 * (7 - POSITION_FOURIER_MIN) / (POSITION_FOURIER_MAX - POSITION_FOURIER_MIN) - 1
)

assert timed["position_mask"].all()
assert (timed["position_values"] == normalized_time).all()
assert not untimed["position_mask"].any()
assert (timed["token_ids"] == untimed["token_ids"]).all()

print(
    {
        key: timed[key].tolist()
        for key in ("token_ids", "position_values", "position_mask")
    }
)

## Build Encoder and Heads

`position_num_frequencies` enables a learned projection for position Fourier features.
Pass both position arrays from `tokens.to(device)` to the embedding. The delayed encoder
includes the same projection and follows the online weights.

Numeric input fields each own a separate learned projection, reused across all
occurrences of that field. The position uses one shared projection across all tokens and
named embedding tables. Both additions happen before the transformer. This notebook uses
text tokens with positions; numeric tokens can use both kinds of projection, with
separate bounds for each scalar.


In [ ]:
class Encoder(Model):
    def __init__(
        self, *, pretrained, tokenizer_sources, dtype
    ):
        super().__init__()
        self.embedding = MultiEmbedding(
            embeddings={
                name: Embedding(pretrained=source)
                for name, source in tokenizer_sources.items()
            },
            position_num_frequencies=POSITION_NUM_FREQUENCIES,
        )
        self.backbone = Backbone(
            pretrained=pretrained,
            dtype=dtype,
        )

    def forward(self, tokens, *, cache=None, use_cache=False):
        if cache is not None and not use_cache:
            raise ValueError("Passing cache= requires use_cache=True.")
        tensors = tokens.to(next(self.parameters()).device)
        token_ids = tensors["token_ids"]
        numeric_values = tensors["numeric_values"]
        numeric_mask = tensors["numeric_mask"]
        sequence_ids = tensors["sequence_ids"]
        head_token_ids = tensors["head_token_mask"].nonzero(as_tuple=True)[0]
        embed = self.embedding(
            token_ids,
            numeric_values=numeric_values,
            numeric_mask=numeric_mask,
            numeric_field_ids=tensors["numeric_field_ids"],
            numeric_field_names=tokens["numeric_field_names"],
            position_values=tensors["position_values"],
            position_mask=tensors["position_mask"],
            tokenizer_ids=tensors["tokenizer_ids"],
            tokenizer_names=tokens["tokenizer_names"],
        )
        if use_cache:
            embed, lengths, head_token_ids, _ = prepare_decode(
                embeds=embed,
                tokens=tokens,
                sequence_ids=sequence_ids,
                head_token_mask=tensors["head_token_mask"],
            )
            if cache is None:
                cache = DecodeCache(
                    sessions=(
                        self.backbone.decode_session(
                            batch_size=tokens["num_sequences"]
                        ),
                    )
                )
            if len(cache.sessions) != 1:
                raise ValueError("expected one backbone cache session")
            features = self.backbone(embed, cache=cache.sessions[0], lengths=lengths)
            head_gather_ids = head_token_ids.unsqueeze(-1).expand(
                *head_token_ids.shape,
                features.shape[-1],
            )
            h = features.gather(1, head_gather_ids)
        else:
            features = self.backbone(embed, sequence_ids=sequence_ids)
            h = features[head_token_ids]
        return {
            "z": h,
            "cache": cache,
        }


class QHead(nn.Module):
    def __init__(self, *, hidden_dim, num_actions, num_layers, scale):
        super().__init__()
        self.mlp = MLP(
            in_features=hidden_dim,
            out_features=num_actions,
            hidden_dim=hidden_dim,
            num_layers=num_layers,
            scale=scale,
            propagate_gradient=1.0,
        )

    def forward(self, z):
        return self.mlp(z)


encoder = (
    Encoder(
        tokenizer_sources=tokenizer.tokenizer_sources,
        pretrained=PRETRAINED,
        dtype=torch.float32,
    )
    .train()
    .to(device)
)

q_head = (
    QHead(
        hidden_dim=encoder.backbone.hidden_dim,
        num_actions=MAX_ACTIONS,
        num_layers=HEAD_LAYERS,
        scale=HEAD_SCALE,
    )
    .train()
    .to(device)
)

# A checkpoint container; forward calls use the encoder and heads separately.
modules = nn.ModuleDict({"encoder": encoder, "q_head": q_head})

delayed_encoder = deepcopy(encoder).requires_grad_(False).eval()
backup_value_head = deepcopy(q_head).requires_grad_(False).eval()

print(encoder)
print(q_head)

## Train

DQN uses one-step decision transitions. This example uses undiscounted returns within
each episode; time embeddings do not implicitly change discounting.

The loader returns `inputs` (tokens, embedding features, sequence IDs, and
`head_token_mask`) and `objective_data` (training labels and step mappings).
`objective_data["head_step_ids"][p]` tells the loss which step owns prediction row `p`,
including steps with zero or multiple predictions. Pass this existing mapping to the
objective; no separate prediction-count column is needed. An absent action is
represented by `NaN` in the packed labels.


In [ ]:
optimizer = AdamW(
    params=[
        *encoder.parameter_groups(
            lr={"embedding": LR_EMBEDDING, "backbone": LR_BACKBONE}
        ),
        {"params": q_head.parameters(), "lr": LR_HEADS},
    ],
    weight_decay=0.0,
    betas=(0.9, 0.95),
    eps=1e-08,
)
objective = DqnObjective(
    reward=None,
    value=None,
    discount=boundary_discount(
        gamma_step=1.0,
        gamma_terminated=0.0,
        gamma_truncated=0.0,
    ),
    temperature=0.0,
    gate=None,
)


def run_train(
    *, num_steps: int
) -> tuple[torch.Tensor, dict[str, float | torch.Tensor]]:
    """Run ``num_steps`` optimizer steps on batches from ``loader``."""
    modules.train()
    loss: torch.Tensor | None = None
    metrics: dict[str, float | torch.Tensor] = {}
    for _ in range(num_steps):
        inputs, objective_data = loader.next_batch()
        inputs = inputs.to(device)
        objective_data = objective_data.to(device)
        group_id = objective_data["step_sequence_ids"]
        out = encoder(inputs)
        with torch.no_grad():
            delayed_out = delayed_encoder(inputs)
            backup_action = q_head(delayed_out["z"])
        losses, metrics = objective(
            objective_data=objective_data.to(device),
            group_id=group_id.to(device),
            head_step_ids=objective_data["head_step_ids"],
            predictions=q_head(out["z"]),
            backup_value_predictions=backup_value_head(delayed_out["z"]),
            backup_action_predictions=backup_action,
        )
        loss = torch.stack(tuple(losses.values())).sum()
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        polyak_update(
            target=delayed_encoder.embedding,
            source=encoder.embedding,
            tau=POLYAK_TAU_EMBEDDING,
        )
        polyak_update(
            target=delayed_encoder.backbone,
            source=encoder.backbone,
            tau=POLYAK_TAU_BACKBONE,
        )
        polyak_update(target=backup_value_head, source=q_head, tau=POLYAK_TAU_VALUE)
    assert loss is not None
    return (loss, metrics)

## Run

This is a short wiring demonstration. Training loss is not a performance evaluation.


In [ ]:
for cycle in range(NUM_CYCLES):
    loss, metrics = run_train(num_steps=TRAIN_STEPS)
    print(
        f"cycle={cycle} train  loss={loss.item():.4f}  q={metrics['q_values_mean']:.3f}"
    )

loader.close()

## Save Locally

Weights include the learned position projection. Rebuild the same encoder architecture
when loading, including `POSITION_NUM_FREQUENCIES`. The saved tokenizer preserves each
field's provider, position field, and scaling bounds. Nothing is uploaded.


In [ ]:
modules.eval().to("cpu")
save_model(model=modules, path=OUTPUT_DIR)
save_tokenizer(tokenizer=tokenizer, path=OUTPUT_DIR)
print(f"Saved model weights and tokenizer fields locally in {OUTPUT_DIR}")